# Multiclass Classification — Build It Yourself

This notebook rebuilds the Course 2, Week 2 "Multiclass Classification" lab, but you write the code.

You already know the architecture (2-layer net, ReLU → Linear, softmax folded into the loss). The goal here is **coding intuition**: what does each line actually do, in terms of arrays and numbers.

**Structure:**
1. Build the same 4-class blob dataset
2. Build + train the Keras model yourself (fill-in-the-blank)
3. Predict with it, and understand *why* the raw output isn't a probability
4. Rebuild the **exact same forward pass in raw NumPy** using the trained weights — no Keras, just `matmul` + `relu` + `softmax`
5. Compare your NumPy output to Keras's output — if they match, you've proven to yourself you understand the whole pipeline

Every TODO has an `assert` right after it. If your cell is wrong, you'll get an `AssertionError`, not a silent bug. Run cells top to bottom.


## 0. Setup

Nothing to fill in here — just imports.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

np.set_printoptions(precision=3, suppress=True)
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)
tf.random.set_seed(1234)
np.random.seed(1234)


## 1. The dataset

Same as the original lab: 4 blobs in 2D, one per class.

**TODO:** create `X_train, y_train` using `make_blobs`.
- `n_samples=100`
- `centers` = `[[-5, 2], [-2, -2], [1, 2], [5, -2]]`
- `cluster_std=1.0`
- `random_state=30`


In [ ]:
classes = 4
m = 100
centers = [[-5, 2], [-2, -2], [1, 2], [5, -2]]
std = 1.0

# TODO: call make_blobs with n_samples=m, the centers/std above, random_state=30
# It returns X_train (features) and y_train (integer class labels)
X_train, y_train = None, None  # <- replace this

# --- checks ---
assert X_train is not None, "You still need to call make_blobs"
assert X_train.shape == (100, 2), f"Expected X_train shape (100,2), got {X_train.shape}"
assert y_train.shape == (100,), f"Expected y_train shape (100,), got {y_train.shape}"
assert set(np.unique(y_train)) == {0, 1, 2, 3}, "Expected 4 classes labeled 0-3"
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("first 10 labels:", y_train[:10])
print("PASS")


**Why this matters:** `y_train` is a 1D array of *integers* (0,1,2,3), not one-hot vectors like `[0,0,1,0]`. This is exactly why the lab uses `SparseCategoricalCrossentropy` instead of plain `CategoricalCrossentropy` — "Sparse" means "labels are given as plain integers, please one-hot them internally for me." That's the whole reason that specific loss function exists.

In [ ]:
plt.figure(figsize=(6,5))
plt.scatter(X_train[:,0], X_train[:,1], c=y_train, cmap='viridis', edgecolors='k')
plt.xlabel('x0'); plt.ylabel('x1')
plt.title('4-class blob dataset')
plt.colorbar(label='class')
plt.show()


## 2. Build the model

Architecture (same as the lab):
- Layer 1 (`L1`): Dense, **2 units**, **ReLU** activation
- Layer 2 (`L2`): Dense, **4 units**, **linear** activation (no activation function)

**Why linear on the output, not softmax?**
Mathematically, `softmax(linear_output)` and having softmax as the layer's own activation give the same final probabilities. But numerically, computing softmax and cross-entropy in *one combined step* is more stable (avoids extreme exp() overflow/underflow). So Keras convention: leave the last layer linear ("logits"), and tell the loss function "hey, these are raw logits, apply softmax yourself" via `from_logits=True`.

**TODO:** build the `Sequential` model with the 2 layers described above.


In [ ]:
tf.random.set_seed(1234)

# TODO: build a Sequential model with:
#   Dense(2, activation='relu', name="L1")
#   Dense(4, activation='linear', name="L2")
model = None  # <- replace this

# --- checks ---
assert model is not None, "You still need to build the model"
model.build(input_shape=(None, 2))
assert len(model.layers) == 2, f"Expected 2 layers, got {len(model.layers)}"
assert model.layers[0].units == 2, "L1 should have 2 units"
assert model.layers[1].units == 4, "L2 should have 4 units (one per class)"
print(model.summary())
print("PASS")


## 3. Compile and train

**TODO:**
- Loss: `tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)`
- Optimizer: `tf.keras.optimizers.Adam(0.01)`
- Then call `.fit()` for 200 epochs on `X_train, y_train`


In [ ]:
# TODO: model.compile(loss=..., optimizer=...)


# TODO: model.fit(X_train, y_train, epochs=200, verbose=0)
history = None  # <- replace this, store fit() return value here

# --- checks ---
assert history is not None, "You still need to call model.fit()"
final_loss = history.history['loss'][-1]
print(f"Final training loss: {final_loss:.4f}")
assert final_loss < 0.15, f"Loss ({final_loss:.4f}) seems too high — check your compile/fit call"
print("PASS — model trained")


In [ ]:
plt.figure(figsize=(6,4))
plt.plot(history.history['loss'])
plt.xlabel('epoch'); plt.ylabel('loss')
plt.title('Training loss')
plt.show()


## 4. Predict — and see the logits problem firsthand

Call `model.predict()` on the training data and look at the raw output.

**TODO:** get predictions, then answer for yourself: do these rows look like probabilities (summing to 1, all between 0 and 1)? They won't — because the output layer is linear. That's expected.


In [ ]:
# TODO: predict on X_train
p_logits = None  # <- replace this

assert p_logits is not None, "Call model.predict(X_train)"
assert p_logits.shape == (100, 4), f"Expected shape (100,4), got {p_logits.shape}"
print("Raw output (logits) for first 3 examples:")
print(p_logits[:3])
print("\nRow sums (NOT 1, because these are logits, not probabilities):")
print(p_logits[:3].sum(axis=1))
print("PASS")


**TODO:** now turn logits into actual probabilities using `tf.nn.softmax()`. This is the step the loss function was doing internally during training — now you do it manually for inference.

In [ ]:
# TODO: apply tf.nn.softmax to p_logits, then convert to a numpy array
p_probs = None  # <- replace this

assert p_probs is not None, "Apply tf.nn.softmax(p_logits) and convert to numpy with .numpy()"
p_probs = np.array(p_probs)
row_sums = p_probs.sum(axis=1)
assert np.allclose(row_sums, 1.0, atol=1e-4), f"Rows should sum to 1 after softmax, got {row_sums[:5]}"
print("Probabilities for first 3 examples:")
print(p_probs[:3])
print("\nRow sums (should all be ~1.0):")
print(row_sums[:3])
print("PASS — softmax correctly turns logits into a probability distribution per row")


**TODO:** get the predicted class for each example — the class with the highest probability (equivalently, highest logit — softmax doesn't change *which one* is biggest, only rescales them, so you could've used `p_logits` too). Use `np.argmax` along the right axis.

In [ ]:
# TODO: predicted class = index of max value in each row of p_probs (axis=1)
y_pred = None  # <- replace this

assert y_pred is not None, "Use np.argmax(p_probs, axis=1)"
accuracy = np.mean(y_pred == y_train)
print(f"Training accuracy: {accuracy*100:.1f}%")
assert accuracy > 0.95, f"Accuracy ({accuracy:.2f}) seems too low — check your model/predictions"
print("PASS")


## 5. The real goal: rebuild the forward pass yourself in raw NumPy

This is the part that actually builds coding intuition. A `Dense` layer is nothing but:

$$z = X W + b$$

then an activation function applied elementwise (or across the row, for softmax).

You'll pull out the **trained weights** from your Keras model and reproduce its exact predictions using only NumPy — no `model.predict()` involved. If your numbers match Keras's, you've proven you understand the whole pipeline down to the arithmetic.

### 5.1 Extract the weights

**TODO:** get `W1, b1` from layer `"L1"` and `W2, b2` from layer `"L2"` using `.get_weights()`.


In [ ]:
# TODO: get layer objects
l1 = None  # model.get_layer("L1")
l2 = None  # model.get_layer("L2")

# TODO: get_weights() returns [W, b] -- unpack both
W1, b1 = None, None
W2, b2 = None, None

assert W1 is not None, "Extract L1's weights"
assert W1.shape == (2, 2), f"W1 should be (2,2) [2 inputs -> 2 units], got {W1.shape}"
assert b1.shape == (2,), f"b1 should be (2,), got {b1.shape}"
assert W2.shape == (2, 4), f"W2 should be (2,4) [2 inputs from L1 -> 4 units], got {W2.shape}"
assert b2.shape == (4,), f"b2 should be (4,), got {b2.shape}"
print("W1 shape:", W1.shape, " b1 shape:", b1.shape)
print("W2 shape:", W2.shape, " b2 shape:", b2.shape)
print("PASS")


**Why these shapes?** `X_train` is `(100, 2)` — 100 examples, 2 features. For `X @ W1` to work, `W1` must be `(2, num_units)`. L1 has 2 units, so `W1` is `(2,2)`. L1's *output* becomes L2's *input* — 2 features again — and L2 has 4 units (one per class), so `W2` is `(2,4)`. This "rows = input dim, columns = number of units" pattern is true for every Dense layer, always.

### 5.2 Write your own `relu` and `softmax` functions

**TODO — relu:** `relu(z) = max(0, z)`, elementwise. Use `np.maximum`.


In [ ]:
def my_relu(z):
    # TODO: return elementwise max(0, z)
    pass

# --- checks ---
test_in = np.array([-2., -0.5, 0., 1.5, 3.])
test_out = my_relu(test_in)
assert test_out is not None, "Implement my_relu"
expected = np.array([0., 0., 0., 1.5, 3.])
assert np.allclose(test_out, expected), f"Expected {expected}, got {test_out}"
print("PASS — my_relu works")


**TODO — softmax:** for a batch of logits `z` with shape `(m, classes)`, softmax per row is:

$$\text{softmax}(z_i)_k = \frac{e^{z_{ik}}}{\sum_j e^{z_{ij}}}$$

i.e. exponentiate every entry, then divide each row by that row's sum. Watch the `axis` and `keepdims=True` — you want to sum *across classes* (columns) for each row (example), and keep the result 2D so broadcasting divides correctly.

Numerical stability tip (used internally by every real implementation): subtract each row's max before exponentiating. It doesn't change the mathematical result (since it cancels in the ratio) but avoids `exp()` overflowing on large logits.


In [ ]:
def my_softmax(z):
    # TODO:
    # 1. subtract the row-wise max from z (for numerical stability): z - z.max(axis=1, keepdims=True)
    # 2. exponentiate
    # 3. divide by the row-wise sum (axis=1, keepdims=True)
    pass

# --- checks ---
test_logits = np.array([[1., 2., 3., 4.], [0., 0., 0., 0.]])
test_probs = my_softmax(test_logits)
assert test_probs is not None, "Implement my_softmax"
assert np.allclose(test_probs.sum(axis=1), 1.0), "Each row must sum to 1"
assert np.allclose(test_probs[1], [0.25, 0.25, 0.25, 0.25]), "Equal logits should give equal (uniform) probabilities"
assert test_probs[0].argmax() == 3, "Highest logit (4.0, index 3) should have highest probability"
print(test_probs)
print("PASS — my_softmax works")


### 5.3 The forward pass, by hand

**TODO:** using `X_train`, `W1`, `b1`, `W2`, `b2`, and your `my_relu` / `my_softmax`, reproduce the full forward pass:

1. `z1 = X_train @ W1 + b1`
2. `a1 = my_relu(z1)`  — this is L1's output, the "new features" the lab talks about
3. `z2 = a1 @ W2 + b2` — L2's raw output (logits)
4. `a2 = my_softmax(z2)` — final probabilities

This is *exactly* what `model.predict()` + `tf.nn.softmax()` did above — except now every step is explicit array math you wrote yourself.


In [ ]:
# TODO: implement steps 1-4 above
z1 = None
a1 = None
z2 = None
a2 = None

# --- checks ---
assert z1 is not None, "Compute z1 = X_train @ W1 + b1"
assert z1.shape == (100, 2), f"z1 should be (100,2), got {z1.shape}"
assert a1 is not None and a1.shape == (100, 2)
assert (a1 >= 0).all(), "ReLU output should never be negative"
assert z2 is not None and z2.shape == (100, 4), f"z2 should be (100,4), got {z2.shape}"
assert a2 is not None and a2.shape == (100, 4)
assert np.allclose(a2.sum(axis=1), 1.0, atol=1e-4), "a2 rows should be probabilities summing to 1"
print("z1 (first 3 rows):\n", z1[:3])
print("a1 (first 3 rows, after relu):\n", a1[:3])
print("z2 (first 3 rows, logits):\n", z2[:3])
print("a2 (first 3 rows, probabilities):\n", a2[:3])
print("PASS")


### 5.4 The moment of truth — does your NumPy match Keras?

If your manual forward pass is correct, `a2` (your NumPy probabilities) should match `p_probs` (Keras's `model.predict()` + `tf.nn.softmax()` from earlier) almost exactly — tiny floating point differences are fine, big differences mean a bug somewhere in your matrix math.


In [ ]:
diff = np.abs(a2 - p_probs)
max_diff = diff.max()
print(f"Max absolute difference between your NumPy output and Keras output: {max_diff:.8f}")
assert max_diff < 1e-3, "Your manual forward pass doesn't match Keras -- check your matmuls, weight shapes, or activation order"

# also check predicted classes match
my_pred = np.argmax(a2, axis=1)
assert np.array_equal(my_pred, y_pred), "Predicted classes differ from Keras's predictions"

print("PASS -- your from-scratch NumPy forward pass exactly reproduces the trained Keras model.")
print("You just proved, with actual numbers, that a neural network layer is nothing but matmul + bias + activation.")


## 6. Visual sanity check

Same decision-boundary style plot as the original lab, done manually: for a grid of points, run them through YOUR forward pass function and color by predicted class.


In [ ]:
def forward(X):
    z1 = X @ W1 + b1
    a1 = my_relu(z1)
    z2 = a1 @ W2 + b2
    a2 = my_softmax(z2)
    return a2

xx, yy = np.meshgrid(np.linspace(-8, 8, 200), np.linspace(-6, 6, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
grid_pred = np.argmax(forward(grid), axis=1).reshape(xx.shape)

plt.figure(figsize=(7,6))
plt.contourf(xx, yy, grid_pred, alpha=0.3, cmap='viridis')
plt.scatter(X_train[:,0], X_train[:,1], c=y_train, cmap='viridis', edgecolors='k')
plt.xlabel('x0'); plt.ylabel('x1')
plt.title('Decision boundaries -- computed entirely by your NumPy forward()')
plt.show()


## Recap

You now have hands-on proof of:

- **`SparseCategoricalCrossentropy`** = "labels are plain integers, one-hot them for me, and I'll softmax the logits internally"
- **`from_logits=True`** = the last Dense layer stays linear; softmax is applied inside the loss (training) or manually via `tf.nn.softmax()` (inference) — purely for numerical stability, mathematically identical to a softmax layer
- **A Dense layer is `X @ W + b`**, full stop. `W`'s shape is always `(inputs, units)`.
- **ReLU** = `max(0, z)`, elementwise.
- **Softmax** = exponentiate + normalize each row so it sums to 1; subtracting the row max first is just for numerical stability and doesn't change the result.
- Chaining these by hand (`X @ W1 + b1 -> relu -> @ W2 + b2 -> softmax`) reproduces Keras's `model.predict()` exactly, because that's *literally all `model.predict()` is doing internally*.

Next time you see a Keras `Sequential` model, you should be able to mentally unroll it into this exact sequence of NumPy operations.
